In [3]:
# First Order Logic - Resolution
# Case Study: Prove that John likes Peanuts.

# A literal is represented as:
# (Negation, Predicate, Arguments)
# True = Negated literal
# False = Positive literal

def is_variable(x):
    return x in ['x', 'y', 'z', 'k']


def apply_substitution(term, subs):
    while term in subs:
        term = subs[term]
    return term


def unify(a, b, subs):
    a = apply_substitution(a, subs)
    b = apply_substitution(b, subs)

    if a == b:
        return True

    if is_variable(a):
        subs[a] = b
        return True

    if is_variable(b):
        subs[b] = a
        return True

    return False


def substitute_literal(literal, subs):
    neg, pred, args = literal
    return (neg, pred,
            tuple(apply_substitution(a, subs) for a in args))


def resolve(c1, c2):
    for l1 in c1:
        for l2 in c2:

            # Check for complementary literals
            if l1[1] == l2[1] and l1[0] != l2[0]:

                if len(l1[2]) != len(l2[2]):
                    continue

                subs = {}
                valid = True

                # Perform unification
                for a, b in zip(l1[2], l2[2]):
                    if not unify(a, b, subs):
                        valid = False
                        break

                if valid:
                    # Remove resolved literals
                    remaining = [
                        l for l in c1 if l != l1
                    ] + [
                        l for l in c2 if l != l2
                    ]

                    # Apply substitution
                    result = [
                        substitute_literal(l, subs)
                        for l in remaining
                    ]

                    return result, subs

    return None, None


def display_clause(clause):
    if not clause:
        return "{}"

    result = []

    for neg, pred, args in clause:
        atom = pred + "(" + ", ".join(args) + ")"

        if neg:
            atom = "NOT " + atom

        result.append(atom)

    return " OR ".join(result)


# ------------------------------------------------
# KNOWLEDGE BASE (CNF)
# ------------------------------------------------

# 1. John likes all kinds of food.
C1 = [(True, "food", ("x",)),
      (False, "likes", ("John", "x"))]

# 2. Apple and vegetables are food.
C2 = [(False, "food", ("Apple",))]
C3 = [(False, "food", ("Vegetables",))]

# 3. Anything anyone eats and is not killed is food.
C4 = [(True, "eats", ("y", "z")),
      (False, "killed", ("y",)),
      (False, "food", ("z",))]

# 4. Anil eats peanuts.
C5 = [(False, "eats", ("Anil", "Peanuts"))]

# 5. Anil is alive.
C6 = [(False, "alive", ("Anil",))]

# 6. Anyone who is alive implies not killed.
# alive(x) -> NOT killed(x)
C7 = [(True, "alive", ("k",)),
      (True, "killed", ("k",))]

# 7. Anyone who is not killed implies alive.
# NOT killed(x) -> alive(x)
C8 = [(False, "killed", ("z",)),
      (False, "alive", ("z",))]

# 8. Harry eats everything that Anil eats.
# eats(Anil, x) -> eats(Harry, x) => NOT eats(Anil, x) OR eats(Harry, x)
C9 = [(True, "eats", ("Anil", "x")),
      (False, "eats", ("Harry", "x"))]


# ------------------------------------------------
# NEGATE THE QUERY
# ------------------------------------------------

# Query: John likes peanuts.
# Negated query: NOT likes(John, Peanuts)

G = [(True, "likes", ("John", "Peanuts"))]


# ------------------------------------------------
# PROOF BY RESOLUTION
# ------------------------------------------------

print("KNOWLEDGE BASE")
print("---------------------")

for i, clause in enumerate(
    [C1, C2, C3, C4, C5, C6, C7, C8, C9], 1
):
    print(f"C{i}: {display_clause(clause)}")

print("\nNegated Goal:")
print(display_clause(G))


print("\n\nRESOLUTION STEPS")
print("---------------------")


# Step 1: Resolve negated goal with C1
R1, S1 = resolve(G, C1)

print("\nStep 1:")
print("Resolving:")
print(display_clause(G))
print(display_clause(C1))
print("Substitution:", S1)
print("Resolvent:", display_clause(R1))


# Step 2: Resolve R1 with C4
R2, S2 = resolve(R1, C4)

print("\nStep 2:")
print("Resolving:")
print(display_clause(R1))
print(display_clause(C4))
print("Substitution:", S2)
print("Resolvent:", display_clause(R2))


# Step 3: Resolve R2 with C5
R3, S3 = resolve(R2, C5)

print("\nStep 3:")
print("Resolving:")
print(display_clause(R2))
print(display_clause(C5))
print("Substitution:", S3)
print("Resolvent:", display_clause(R3))


# Step 4: Resolve R3 with C7
R4, S4 = resolve(R3, C7)

print("\nStep 4:")
print("Resolving:")
print(display_clause(R3))
print(display_clause(C7))
print("Substitution:", S4)
print("Resolvent:", display_clause(R4))


# Step 5: Resolve R4 with C6
R5, S5 = resolve(R4, C6)

print("\nStep 5:")
print("Resolving:")
print(display_clause(R4))
print(display_clause(C6))
print("Substitution:", S5)
print("Resolvent:", display_clause(R5))


# ------------------------------------------------
# FINAL RESULT
# ------------------------------------------------

print("\n\nFINAL RESULT")
print("---------------------")

if R5 == []:
    print("Empty clause {} obtained.")
    print("Contradiction found!")
    print("Hence, John likes Peanuts is PROVED.")
else:
    print("Proof could not be completed.")

KNOWLEDGE BASE
---------------------
C1: NOT food(x) OR likes(John, x)
C2: food(Apple)
C3: food(Vegetables)
C4: NOT eats(y, z) OR killed(y) OR food(z)
C5: eats(Anil, Peanuts)
C6: alive(Anil)
C7: NOT alive(k) OR NOT killed(k)
C8: killed(z) OR alive(z)
C9: NOT eats(Anil, x) OR eats(Harry, x)

Negated Goal:
NOT likes(John, Peanuts)


RESOLUTION STEPS
---------------------

Step 1:
Resolving:
NOT likes(John, Peanuts)
NOT food(x) OR likes(John, x)
Substitution: {'x': 'Peanuts'}
Resolvent: NOT food(Peanuts)

Step 2:
Resolving:
NOT food(Peanuts)
NOT eats(y, z) OR killed(y) OR food(z)
Substitution: {'z': 'Peanuts'}
Resolvent: NOT eats(y, Peanuts) OR killed(y)

Step 3:
Resolving:
NOT eats(y, Peanuts) OR killed(y)
eats(Anil, Peanuts)
Substitution: {'y': 'Anil'}
Resolvent: killed(Anil)

Step 4:
Resolving:
killed(Anil)
NOT alive(k) OR NOT killed(k)
Substitution: {'k': 'Anil'}
Resolvent: NOT alive(Anil)

Step 5:
Resolving:
NOT alive(Anil)
alive(Anil)
Substitution: {}
Resolvent: {}


FINAL RESULT
--